In [1]:
# ==============================================================
# 10_classification_blending.ipynb
# --------------------------------------------------------------
# Blends multiple classifiers (Logistic, DecisionTree, RandomForest, SVM)
# using both simple and weighted soft-voting ensembles
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# ==============================================================
# Configuration
# ==============================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_enhanced_model_ready.csv",
    "TCS": data_dir / "tcs_enhanced_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_enhanced_model_ready.csv",
}

# ==============================================================
# Helper
# ==============================================================
def evaluate_classification(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba)
    }

# ==============================================================
# Main Execution
# ==============================================================
all_results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"⚠️ Missing: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded dataset shape: {df.shape}")

    target_col = "Target_Cls"
    if target_col not in df.columns:
        print(f"⚠️ Skipping {ticker} — Target_Cls not found.")
        continue

    df = df.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].dropna(axis=1, how="all")

    imputer = SimpleImputer(strategy="mean")
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col, "Target_Reg"], errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    models = {
        "Logistic": LogisticRegression(max_iter=1000),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=150, random_state=42),
        "SVM": SVC(kernel='rbf', probability=True)
    }

    preds_dict = {}
    metrics_dict = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba > 0.5).astype(int)

        metrics = evaluate_classification(y_test, y_pred, y_proba)
        metrics.update({"Ticker": ticker, "Model": name})
        all_results.append(metrics)

        preds_dict[name] = y_proba
        metrics_dict[name] = metrics["F1"]

        print(f"  → {name}: Acc={metrics['Accuracy']:.3f}, F1={metrics['F1']:.3f}, Prec={metrics['Precision']:.3f}, Rec={metrics['Recall']:.3f}")

    # --- Simple Blend ---
    blend_simple = np.mean(list(preds_dict.values()), axis=0)
    y_pred_simple = (blend_simple > 0.5).astype(int)
    metrics_simple = evaluate_classification(y_test, y_pred_simple, blend_simple)
    metrics_simple.update({"Ticker": ticker, "Model": "Blend_Simple"})
    all_results.append(metrics_simple)
    print(f"  ✅ Blend_Simple: Acc={metrics_simple['Accuracy']:.3f}, F1={metrics_simple['F1']:.3f}")

    # --- Weighted Blend (based on F1) ---
    weights = np.array(list(metrics_dict.values()))
    weights = weights / weights.sum() if weights.sum() != 0 else np.ones(len(weights)) / len(weights)
    blend_weighted = np.average(list(preds_dict.values()), axis=0, weights=weights)
    y_pred_weighted = (blend_weighted > 0.5).astype(int)
    metrics_weighted = evaluate_classification(y_test, y_pred_weighted, blend_weighted)
    metrics_weighted.update({"Ticker": ticker, "Model": "Blend_Weighted"})
    all_results.append(metrics_weighted)
    print(f"  ✅ Blend_Weighted: Acc={metrics_weighted['Accuracy']:.3f}, F1={metrics_weighted['F1']:.3f}")

# ==============================================================
# Save Results
# ==============================================================
results_df = pd.DataFrame(all_results)
save_path = results_dir / "classification_blending_results.csv"
results_df.to_csv(save_path, index=False)

print(f"\n✅ Classification Blending completed. Results saved to: {save_path}")
display(results_df.sort_values(["Ticker", "F1"], ascending=[True, False]))



=== Processing RELIANCE ===
  Loaded dataset shape: (1460, 25)
  → Logistic: Acc=0.483, F1=0.488, Prec=0.514, Rec=0.465
  → DecisionTree: Acc=0.500, F1=0.397, Prec=0.552, Rec=0.310
  → RandomForest: Acc=0.497, F1=0.269, Prec=0.587, Rec=0.174
  → SVM: Acc=0.531, F1=0.694, Prec=0.531, Rec=1.000
  ✅ Blend_Simple: Acc=0.500, F1=0.397
  ✅ Blend_Weighted: Acc=0.500, F1=0.397

=== Processing TCS ===
  Loaded dataset shape: (1460, 25)
  → Logistic: Acc=0.503, F1=0.551, Prec=0.497, Rec=0.618
  → DecisionTree: Acc=0.500, F1=0.180, Prec=0.471, Rec=0.111
  → RandomForest: Acc=0.500, F1=0.151, Prec=0.464, Rec=0.090
  → SVM: Acc=0.493, F1=0.661, Prec=0.493, Rec=1.000
  ✅ Blend_Simple: Acc=0.500, F1=0.180
  ✅ Blend_Weighted: Acc=0.500, F1=0.180

=== Processing HDFCBANK ===
  Loaded dataset shape: (1460, 25)
  → Logistic: Acc=0.514, F1=0.601, Prec=0.535, Rec=0.686
  → DecisionTree: Acc=0.510, F1=0.557, Prec=0.539, Rec=0.577
  → RandomForest: Acc=0.500, F1=0.486, Prec=0.539, Rec=0.442
  → SVM: Acc=0.5

,Accuracy,Precision,Recall,F1,ROC_AUC,Ticker,Model
15,0.534247,0.534247,1.000000,0.696429,0.569005,HDFCBANK,SVM
12,0.513699,0.535000,0.685897,0.601124,0.488216,HDFCBANK,Logistic
13,0.510274,0.538922,0.576923,0.557276,0.505373,HDFCBANK,DecisionTree
16,0.510274,0.538922,0.576923,0.557276,0.509898,HDFCBANK,Blend_Simple
17,0.510274,0.538922,0.576923,0.557276,0.509333,HDFCBANK,Blend_Weighted
14,0.500000,0.539062,0.442308,0.485915,0.524086,HDFCBANK,RandomForest
3,0.530822,0.530822,1.000000,0.693512,0.503697,RELIANCE,SVM
0,0.482877,0.514286,0.464516,0.488136,0.481846,RELIANCE,Logistic
1,0.500000,0.551724,0.309677,0.396694,0.512503,RELIANCE,DecisionTree
4,0.500000,0.551724,0.309677,0.396694,0.516600,RELIANCE,Blend_Simple
